In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Feedforward clássico e fluxo de controle (circuitos dinâmicos)

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



# Feedforward clássico e fluxo de controle


<details>
<summary><b>Versões dos pacotes</b></summary>

O código nesta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar estas versões ou mais recentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
> **Note:** A nova versão dos circuitos dinâmicos está agora disponível para todos os usuários em todos os backends. Você pode executar circuitos dinâmicos em escala de utilidade. Veja [o anúncio](/announcements/product-updates/2025-09-25-new-dynamic-circuits) para mais detalhes.

Os circuitos dinâmicos são ferramentas poderosas com as quais você pode medir qubits no meio da execução de um Circuit quântico e, em seguida, realizar operações de lógica clássica dentro do circuito, com base nos resultados dessas medições intermediárias. Esse processo também é conhecido como _feedforward clássico_. Embora ainda seja cedo para entender como melhor aproveitar os circuitos dinâmicos, a comunidade de pesquisa quântica já identificou vários casos de uso, como os seguintes:

* Preparação eficiente de estados quânticos, como o [estado GHZ,](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) [estado W,](https://arxiv.org/abs/2403.07604) (para mais informações sobre o estado W, consulte também ["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)) e uma ampla classe de [estados de produto matricial](https://arxiv.org/abs/2404.16083)
* [Emaranhamento eficiente de longo alcance](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) entre qubits no mesmo chip usando circuitos rasos
* [Amostragem eficiente de circuitos similares a IQP](https://arxiv.org/pdf/2505.04705)

Essas melhorias trazidas pelos circuitos dinâmicos, no entanto, vêm com compensações. As medições intermediárias e as operações clássicas normalmente têm um tempo de execução maior do que as gates de dois qubits, e esse aumento no tempo pode anular os benefícios da redução da profundidade do circuito. Portanto, reduzir a duração das medições intermediárias é uma área de foco de melhoria à medida que a IBM Quantum&reg; lança a [nova versão](/announcements/product-updates/2025-03-03-new-version-dynamic-circuits) dos circuitos dinâmicos.

A [especificação OpenQASM 3](https://openqasm.com/language/classical.html#looping-and-branching) define diversas estruturas de fluxo de controle, mas o Qiskit Runtime atualmente suporta apenas a instrução condicional `if`. No Qiskit SDK, isso corresponde ao método [if_test](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test) no [QuantumCircuit.](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) Esse método retorna um [gerenciador de contexto](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers) e normalmente é usado em uma instrução `with`. Este guia descreve como usar essa instrução condicional.

> **Note:** Os exemplos de código neste guia usam a instrução de medição padrão para medições intermediárias. No entanto, recomenda-se usar a instrução [`MidCircuitMeasure`](/guides/measure-qubits#midcircuit) em vez disso, se o backend oferecer suporte. Consulte a [documentação de medições intermediárias](/guides/measure-qubits#mid-circuit-measurements) para mais detalhes.
## Instrução `if`
A instrução `if` é usada para realizar operações condicionalmente com base no valor de um bit ou registrador clássico.

No exemplo abaixo, aplicamos uma gate Hadamard a um qubit e o medimos. Se o resultado for 1, aplicamos uma gate X no qubit, que tem o efeito de revertê-lo para o estado 0. Em seguida, medimos o qubit novamente. O resultado da medição deve ser 0 com 100% de probabilidade.

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
# Use MidCircuitMeasure() if it's supported by the backend.
# circuit.append(MidCircuitMeasure(), [q0], [c0])
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg" alt="Output of the previous code cell" />

![Saída da célula de código anterior](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg)

A instrução `with` pode receber um alvo de atribuição que é, por si só, um gerenciador de contexto que pode ser armazenado e posteriormente usado para criar um bloco else, que é executado sempre que o conteúdo do bloco `if` *não* é executado.

No exemplo abaixo, inicializamos registradores com dois qubits e dois bits clássicos. Aplicamos uma gate Hadamard ao primeiro qubit e o medimos. Se o resultado for 1, aplicamos uma gate Hadamard no segundo qubit; caso contrário, aplicamos uma gate X no segundo qubit. Por fim, medimos o segundo qubit também.

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg" alt="Output of the previous code cell" />

![Saída da célula de código anterior](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg)

Além de condicionar em um único bit clássico, também é possível condicionar no valor de um registrador clássico composto por múltiplos bits.

No exemplo abaixo, aplicamos gates Hadamard a dois qubits e os medimos. Se o resultado for `01`, ou seja, o primeiro qubit é 1 e o segundo qubit é 0, então aplicamos uma gate X a um terceiro qubit. Por fim, medimos o terceiro qubit. Observe que, para maior clareza, optamos por especificar o estado do terceiro bit clássico, que é 0, na condição `if`. No desenho do circuito, a condição é indicada pelos círculos nos bits clássicos sobre os quais a condição é aplicada. Um círculo preto indica condicionamento em 1, enquanto um círculo branco indica condicionamento em 0.

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg" alt="Output of the previous code cell" />

![Saída da célula de código anterior](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg)

## Expressões clássicas
O módulo de expressões clássicas do Qiskit [`qiskit.circuit.classical`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical) contém uma representação exploratória de operações em tempo de execução sobre valores clássicos durante a execução do circuito. Devido a limitações de hardware, apenas as condições `QuantumCircuit.if_test()` são atualmente suportadas.

O exemplo a seguir mostra que você pode usar o cálculo de paridade para criar um estado GHZ de n qubits usando circuitos dinâmicos. Primeiro, gere $n/2$ pares de Bell em qubits adjacentes. Em seguida, una esses pares usando uma camada de gates CNOT entre os pares. Você então mede o qubit alvo de todas as gates CNOT anteriores e reinicia cada qubit medido para o estado $\vert 0 \rangle$. Você aplica $X$ a cada site não medido para o qual a paridade de todos os bits precedentes é ímpar. Por fim, as gates CNOT são aplicadas aos qubits medidos para restabelecer o emaranhamento perdido na medição.

No cálculo de paridade, o primeiro elemento da expressão construída envolve elevar o objeto Python `mr[0]` a um nó [`Value`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical#value) (`lift` é usado para transformar objetos arbitrários em expressões clássicas). Isso não é necessário para `mr[1]` e o possível registrador clássico seguinte, pois eles são entradas para `expr.bit_xor`, e qualquer elevação necessária é feita automaticamente nesses casos. Tais expressões podem ser construídas em loops e outras estruturas.

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [5]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg" alt="Output of the previous code cell" />

![Saída da célula de código anterior](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg)

## Encontrar backends que suportam circuitos dinâmicos
Para encontrar todos os backends que sua conta pode acessar e que suportam circuitos dinâmicos, execute um código como o seguinte. Este exemplo assume que você [salvou suas credenciais de login.](/guides/save-credentials) Você também pode [especificar credenciais explicitamente](/guides/initialize-account#explicit) ao inicializar sua conta de serviço do Qiskit Runtime. Isso permitiria, por exemplo, visualizar os backends disponíveis em uma instância ou tipo de plano específico.

> **Note:** - Os backends disponíveis para a conta dependem da instância especificada nas credenciais.
> - A nova versão dos circuitos dinâmicos está agora disponível para todos os usuários em todos os backends. Veja [o anúncio](/announcements/product-updates/2025-09-25-new-dynamic-circuits) para mais detalhes.

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
dc_backends = service.backends(dynamic_circuits=True)
print(dc_backends)

[<IBMBackend('ibm_pittsburgh')>, <IBMBackend('ibm_boston')>, <IBMBackend('ibm_fez')>, <IBMBackend('ibm_miami')>, <IBMBackend('ibm_marrakesh')>, <IBMBackend('ibm_torino')>, <IBMBackend('ibm_kingston')>]


## Qiskit Runtime limitations

Be aware of the following constraints when running dynamic circuits in Qiskit Runtime.

- Due to the limited physical memory on control electronics, there is also a limit on the number of `if` statements and size of their operands. This limit is a function of the number of broadcasts and the number of broadcasted bits in a job (not a circuit).

   When processing an `if` condition, measurement data needs to be transferred to the control logic to make that evaluation. A broadcast is a transfer of unique classical data, and broadcasted bits is the number of classical bits being transferred. Consider the following:

   ```python
   c0 = ClassicalRegister(3)
   c1 = ClassicalRegister(5)
   ...
   with circuit.if_test((c0, 1)) ...
   with circuit.if_test((c0, 3)) ...
   with circuit.if_test((c1[2], 1)) ...
   ```
   In the previous code example, the first two `if_test` objects on `c0` are considered one broadcast because the content of `c0` did not change, and thus does not need to be re-broadcasted. The `if_test` on `c1` is a second broadcast. The first one broadcasts all three bits in `c0` and the second broadcasts just one bit, making a total of four broadcasted bits.

   Currently, if you broadcast 60 bits each time, then the job can have approximately 300 broadcasts. If you broadcast just one bit each time, however, then the job can have 2400 broadcasts.

- The operand used in an `if_test` statement must be 32 or fewer bits. Thus, if you are comparing an entire `ClassicalRegister`, the size of that `ClassicalRegister` must be 32 or fewer bits. If you are comparing just a single bit from a `ClassicalRegister`, however, that `ClassicalRegister` can be of any size (since the operand is only one bit).

   For example, the "Not valid" code block does not work because `cr` is more than 32 bits.  You can, however, use a classical register wider than 32 bits if you are testing only one bit, as shown in the "Valid" code block.

   <Tabs>
   <TabItem value="Not valid" label="Not valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr, 15)):
            ...
      ```
   </TabItem>
   <TabItem value="Valid" label="Valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr[5], 1)):
            ...
      ```
   </TabItem>
   </Tabs>

- Nested conditionals are not allowed. For example, the following code block will not work because it has an `if_test` inside another `if_test`:
   <Tabs>
    <TabItem value="Not valid" label="Not valid">
        ```python
           c1 = ClassicalRegister(1, "c1")
           c2 = ClassicalRegister(2, "c2")
           ...
           with circ.if_test((c1, 1)):
            with circ.if_test(c2, 1)):
             ...
        ```
     </TabItem>
     <TabItem value="Valid" label="Valid">
        ```python
        cr = ClassicalRegister(2)
        ...
        with circuit.if_test((cr, 0b11)):
          ...
        ```
     </TabItem>
    </Tabs>

- Having `reset` or measurements inside conditionals is not supported.
- Arithmetic operations are not supported.
- See the [OpenQASM 3 feature table](/docs/guides/qasm-feature-table) to determine which OpenQASM 3 features are supported on Qiskit and Qiskit Runtime.
- When OpenQASM 3 (instead of `QuantumCircuit`) is used as the input format to pass circuits to Qiskit Runtime primitives, only instructions that can be loaded into Qiskit are supported. Classical operations, for example, are not supported because they cannot be loaded into Qiskit. See [Import an OpenQASM 3 program into Qiskit](/docs/guides/interoperate-qiskit-qasm3#import-an-openqasm-3-program-into-qiskit) for more information.
- The `for`, `while`, and `switch` instructions are not supported.

## Use dynamic circuits with Estimator

Since Estimator does not support dynamic circuits, you can use Sampler and build your own measurement circuits instead. Alternatively, you can use the [Executor primitive,](/docs/guides/directed-execution-model#executor-primitive) which supports dynamic circuits.

To replicate Estimator's behavior, follow this process:

1. Group the terms of all observables into a partition.  This can be done by using the [`PauliList` API,](/docs/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting) for example.
     <Admonition type="note">
      You can use the [`BitArray`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.BitArray#expectation_values) primitive attribute to compute the expectation values of the provided observables.
     </Admonition>
2. Execute one basis change circuit per partition (whichever basis change needs to be done for each partition). See the Measurement bases addon utility  [`measurement_bases` module](https://github.com/Qiskit/qiskit-addon-utils/blob/38ea05431f2e9830adf4ec9265f6d19758a32096/qiskit_addon_utils/exp_vals/measurement_bases.py) for more information. [Get started with utilities.](/docs/guides/qiskit-addons-utils#get-started-with-utilities)
3. Add back together the results for each partition.

## Next steps

<Admonition type="tip" title="Recommendations">
- Learn how to implement accurate dynamic decoupling by using [stretch.](/docs/guides/stretch)
- Learn about the shorter [mid-circuit measurements](/docs/guides/measure-qubits#mid-circuit-measurements) that reduce the circuit time.
- Use [circuit schedule visualization](/docs/guides/visualize-circuit-timing#qiskit-runtime-support) to debug and optimize your dynamic circuits.
</Admonition>